In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.models_stats import prepare_statsforecast_df, make_model_list, compute_metrics
from statsforecast import StatsForecast

data = pd.read_parquet('../data/processed/ca1_foods3.parquet')
train_df = prepare_statsforecast_df(data, 'train')
valid_df = prepare_statsforecast_df(data, 'val')
 
ids = train_df['unique_id'].drop_duplicates().sample(50, random_state=42).tolist()
train_sample = train_df[train_df['unique_id'].isin(ids)]
valid_sample = valid_df[valid_df['unique_id'].isin(ids)]
print('sample size:', len(ids))

for model in make_model_list():
    sf = StatsForecast(models=[model], freq='D')
    sf.fit(train_sample)
    fc = sf.predict(h=28)
    result = compute_metrics(valid_sample, fc, model.alias)
    print(model.alias, result)


# Bước 4: Baseline thống kê với StatsForecast

Mục tiêu là kiểm tra pipeline trên mẫu 50 chuỗi, xác định mô hình phù hợp và đánh giá trên validation theo MASE/WAPE.
